# GDP AND INFLATION DATA COLLECTION

In this notebook we are going to collect the necessary data to create the normalized anual GDP per capita for the United States. The GDP per capita data is from: https://datosmacro.expansion.com/pib/usa, and the inflation data is from: https://datos.bancomundial.org/indicador/FP.CPI.TOTL.ZG?end=2024&locations=US&start=2010

In [1]:
import pandas as pd
from io import StringIO

First we preprocess the inflation data:

In [2]:
df_inflation = pd.read_csv("../../Data/inflation.csv", skiprows=4)

In [3]:
df_inflation.head()

,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,...,2017,2018,2019,2020,2021,2022,2023,2024,2025,Unnamed: 70
0,Aruba,ABW,"Inflación, precios al consumidor (% anual)",FP.CPI.TOTL.ZG,NaN,NaN,NaN,NaN,NaN,NaN,...,-1.028282,3.626041,4.257462,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,AFE,"Inflación, precios al consumidor (% anual)",FP.CPI.TOTL.ZG,NaN,NaN,NaN,NaN,NaN,NaN,...,6.221375,4.689806,4.102851,5.191629,6.824727,10.883478,7.399186,4.770857,NaN,NaN
2,Afganistán,AFG,"Inflación, precios al consumidor (% anual)",FP.CPI.TOTL.ZG,NaN,NaN,NaN,NaN,NaN,NaN,...,4.975952,0.626149,2.302373,5.601888,5.133203,13.712102,-4.644709,-6.601186,NaN,NaN
3,NaN,AFW,"Inflación, precios al consumidor (% anual)",FP.CPI.TOTL.ZG,NaN,NaN,NaN,NaN,NaN,NaN,...,1.725486,1.784050,1.983092,2.490378,3.745568,7.949251,5.221168,3.608044,NaN,NaN
4,Angola,AGO,"Inflación, precios al consumidor (% anual)",FP.CPI.TOTL.ZG,NaN,NaN,NaN,NaN,NaN,NaN,...,29.844480,19.628938,17.080954,22.271539,25.754295,21.355290,13.644102,28.240495,NaN,NaN


In [4]:
# Create a list of the years that we want to keep (2010-2022)
desired_years = [str(year) for year in range(2010, 2023)]
cols_to_delete = [col for col in desired_years if col in df_inflation.columns]

# Filter to keep only USA data
df_usa = df_inflation[df_inflation['Country Code'] == 'USA']

# Keep only the columns for the years we want to keep
df_usa_filtered = df_usa[cols_to_delete]
df_final = df_usa_filtered.melt(var_name='year', value_name='inflation')

# Convert the 'year' column to integer type
df_final['year'] = df_final['year'].astype(int)

In [5]:
df_final.head(15)

,year,inflation
0,2010,1.640043
1,2011,3.156842
2,2012,2.069337
3,2013,1.464833
4,2014,1.622223
5,2015,0.118627
6,2016,1.261583
7,2017,2.130110
8,2018,2.442583
9,2019,1.812210


Now we are preprocessing the GDP per capita data:

In [6]:
gdp_data = """year,Nominal_GDP
2022,73964
2021,60299
2020,56531
2019,58558
2018,53462
2017,53390
2016,52575
2015,51375
2014,41588
2013,40179
2012,40243"""

df_gdp = pd.read_csv(StringIO(gdp_data))

# Merge the GDP data with the inflation data on the 'year' column
df_mix_gdp = pd.merge(df_gdp, df_final, on='year')

# We sort by year from oldest to newest
df_mix_gdp = df_mix_gdp.sort_values('year').reset_index(drop=True)

# We calculate the accumulated inflation forward
df_mix_gdp['Multiplier'] = 1 + (df_mix_gdp['inflation'] / 100)
df_mix_gdp['Cumulative_Inflation_Index'] = df_mix_gdp['Multiplier'].cumprod()

# We calculate the normalized GDP by dividing the nominal GDP by the cumulative inflation index
df_mix_gdp['Normalized_GDP'] = df_mix_gdp['Nominal_GDP'] / df_mix_gdp['Cumulative_Inflation_Index']
df_mix_gdp['Normalized_GDP'] = df_mix_gdp['Normalized_GDP'].round(0).astype(int)

df_mix_gdp[['year', 'Nominal_GDP', 'inflation', 'Normalized_GDP']].head(15)

,year,Nominal_GDP,inflation,Normalized_GDP
0,2012,40243,2.069337,39427
1,2013,40179,1.464833,38796
2,2014,41588,1.622223,39516
3,2015,51375,0.118627,48757
4,2016,52575,1.261583,49274
5,2017,53390,2.130110,48994
6,2018,53462,2.442583,47891
7,2019,58558,1.812210,51522
8,2020,56531,1.233584,49132
9,2021,60299,4.697859,50056


Finally we save the data:

In [7]:
df_mix_gdp.to_csv("../../Data/normalized_gdp.csv", index=False)